# AAPL News Coverage Validation

Validate the article-count logic for the paper's news filter on one stock:

1. Pull Refinitiv headlines for AAPL over 2003-2014.
2. Count articles per calendar day.
3. Aggregate to weekly counts.
4. Check whether AAPL passes the paper rule: average articles per week >= 1.

Keep LSEG Workspace open when using desktop session mode.

In [ ]:
from pathlib import Path
import sys

import plotly.express as px
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
load_dotenv(PROJECT_ROOT / ".env")

from sentiment_ltr.data.news_coverage import (
    daily_article_counts,
    fetch_ticker_headlines,
    summarize_news_coverage,
    weekly_article_counts,
)

TICKER = "AAPL"
START = "2003-01-01"
END = "2014-12-31"

In [ ]:
headlines, ric = fetch_ticker_headlines(PROJECT_ROOT, TICKER, START, END)
daily = daily_article_counts(headlines, START, END)
weekly = weekly_article_counts(daily)
summary = summarize_news_coverage(TICKER, ric, headlines, START, END)

print(f"RIC: {ric}")
print(f"Headlines: {len(headlines):,}")
print(f"Unique stories: {summary.total_articles:,}")
print(f"Average articles/week: {summary.avg_articles_per_week:.2f}")
print(f"Paper threshold pass: {summary.passes_paper_weekly_threshold}")
daily.tail()

In [ ]:
plot_daily = daily.copy()
plot_daily = plot_daily[plot_daily["article_count"] > 0]

fig = px.bar(
    plot_daily,
    x="date",
    y="article_count",
    title=f"{TICKER} Refinitiv articles per day ({START} to {END})",
    labels={"date": "Date", "article_count": "Articles"},
    hover_data={"date": True, "article_count": True},
)
fig.update_layout(hovermode="closest")
fig.show()